In [0]:
"""
id: python_3
template: python
templateVersion: 1.0.0
name: API_fetchData
position:
  x: 630
  y: 602.2934283994554
description:
  text: Load customer data from a URL or use provided input data.
  hash: 5c5612c9
previewCodeHash: 4f36f1562d7d4c9d
previewMode: "1000"
config:
  code: |
    if inputs.get("data"):
        result = inputs["data"][0]
    else:
        import pandas as pd
        import requests
        from io import StringIO
        url = "https://raw.githubusercontent.com/anshlambagit/Databricks_Lakeflow_Designer/refs/heads/main/customers.csv"

        response = requests.get(url)
        csv_data = response.content.decode('utf-8')
        df = pd.read_csv(StringIO(csv_data))
        result = spark.createDataFrame(df) 
input: []
"""

# generated from the system
from typing import Dict, Any
from pyspark.sql import DataFrame

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    data = inputs.get("data", [] if True else None)
    result = data[0] if data else spark.createDataFrame([], "col: string")

    if inputs.get("data"):
        result = inputs["data"][0]
    else:
        import pandas as pd
        import requests
        from io import StringIO
        url = "https://raw.githubusercontent.com/anshlambagit/Databricks_Lakeflow_Designer/refs/heads/main/customers.csv"

        response = requests.get(url)
        csv_data = response.content.decode('utf-8')
        df = pd.read_csv(StringIO(csv_data))
        result = spark.createDataFrame(df)

    return {"result": result}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {}
inputs = {}
out = run(config, inputs, spark)
ctx["python_3.result"] = out["result"]

In [0]:
"""
id: python_4
template: python
templateVersion: 1.0.0
name: API_fetchdata2
position:
  x: 907
  y: 695.875
description:
  text: Load shipment data from a web CSV file or use provided data.
  hash: 3dfcaa50
previewCodeHash: 0569f63e8b05448d
previewMode: "1000"
config:
  code: |+
    if inputs.get("data"):
        result = inputs["data"][0]
    else:
        import pandas as pd
        import requests
        from io import StringIO
        
        url = "https://raw.githubusercontent.com/anshlambagit/Databricks_Lakeflow_Designer/refs/heads/main/shipments.csv"

        response = requests.get(url)
        csv_data = response.content.decode('utf-8')
        df = pd.read_csv(StringIO(csv_data))
        result = spark.createDataFrame(df)
         
         
         
input: []
"""

# generated from the system
from typing import Dict, Any
from pyspark.sql import DataFrame

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    data = inputs.get("data", [] if True else None)
    result = data[0] if data else spark.createDataFrame([], "col: string")

    if inputs.get("data"):
        result = inputs["data"][0]
    else:
        import pandas as pd
        import requests
        from io import StringIO

        url = "https://raw.githubusercontent.com/anshlambagit/Databricks_Lakeflow_Designer/refs/heads/main/shipments.csv"

        response = requests.get(url)
        csv_data = response.content.decode('utf-8')
        df = pd.read_csv(StringIO(csv_data))
        result = spark.createDataFrame(df)

    return {"result": result}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {}
inputs = {}
out = run(config, inputs, spark)
ctx["python_4.result"] = out["result"]

In [0]:
"""
id: source_0
template: source
templateVersion: 2.0.0
name: orders
position:
  x: 320.94523666212814
  y: 407.89123786594047
description:
  text: Load all data from the orders table.
  hash: 4c02a710
previewCodeHash: 4a03215ce58c6c94
previewMode: "1000"
config:
  table_source:
    tableName: lakeflow_designer.bronze_tables.orders
input: []
"""

# generated from the system
from typing import Dict, Any, List

def _strip_sql_quotes(s):
    if isinstance(s, str) and len(s) >= 2:
        if (s[0] == '"' and s[-1] == '"') or (s[0] == "'" and s[-1] == "'"):
            return s[1:-1]
    return s

def _build_metric_view_sql(
    table_name: str, dims: List[str], measures: List[str]
) -> str:
    def q(n: str) -> str:
        return "`" + n.replace("`", "``") + "`"

    select_parts = [q(d) for d in dims] + [f"MEASURE({q(m)}) AS {q(m)}" for m in measures]
    if not select_parts:
        return f"SELECT * FROM {table_name}"
    sql = f"SELECT {', '.join(select_parts)} FROM {table_name}"
    if dims and measures:
        sql += " GROUP BY " + ", ".join(q(d) for d in dims)
    elif dims and not measures:
        sql = f"SELECT DISTINCT {', '.join(q(d) for d in dims)} FROM {table_name}"
    return sql

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    file_source = config.get("file_source")
    table_source = config.get("table_source")

    if file_source:
        path = file_source.get("path")
        if not path:
            raise ValueError("Source: 'path' is required for file source")
        options = []
        for key, value in file_source.items():
            if key == "path":
                continue
            if key == "headerRows" and isinstance(value, bool):
                options.append(f"{key}=>{1 if value else 0}")
            elif isinstance(value, bool):
                options.append(f'{key}=>{"true" if value else "false"}')
            elif isinstance(value, (int, float)):
                options.append(f"{key}=>{value}")
            else:
                clean = _strip_sql_quotes(str(value))
                if key == "dataAddress" and clean.startswith("!"):
                    clean = clean[1:]
                options.append(f'{key}=>"{clean}"')
        opts = ", ".join(options)
        sql = f'SELECT * FROM read_files("{path}", {opts})' if opts else f'SELECT * FROM read_files("{path}")'
        out = spark.sql(sql)
    elif table_source:
        table_name = table_source.get("tableName")
        if not table_name:
            raise ValueError("Source: 'tableName' is required for table source")

        mv_selection = table_source.get("metricView")
        if mv_selection is not None:
            dims = mv_selection.get("dimensions")
            measures = mv_selection.get("measures")
            if dims is None or measures is None:
                raise ValueError(
                    "Source: metricView selection is incomplete (both "
                    "'dimensions' and 'measures' must be explicit lists). "
                    "Re-open the source node and select the metric view "
                    "again to re-seed the picker."
                )
            sql = _build_metric_view_sql(table_name, list(dims), list(measures))
            out = spark.sql(sql)
        elif table_source.get("isExpression"):
            out = spark.sql(table_name)
        else:
            out = spark.table(table_name)
    else:
        raise ValueError("Source: either 'file_source' or 'table_source' must be configured")

    return {"data": out}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {
    "table_source": {
        "tableName": "lakeflow_designer.bronze_tables.orders"
    }
}
inputs = {}
out = run(config, inputs, spark)
ctx["source_0.data"] = out["data"]

In [0]:
"""
id: source_1
template: source
templateVersion: 2.0.0
name: order_items
position:
  x: 321.15180826267283
  y: 650.4796237482195
description:
  text: Load all data from the order_items table.
  hash: 959edab3
previewCodeHash: 6002d2240ea39263
previewMode: "1000"
config:
  table_source:
    tableName: lakeflow_designer.bronze_tables.order_items
input: []
"""

# generated from the system
from typing import Dict, Any, List

def _strip_sql_quotes(s):
    if isinstance(s, str) and len(s) >= 2:
        if (s[0] == '"' and s[-1] == '"') or (s[0] == "'" and s[-1] == "'"):
            return s[1:-1]
    return s

def _build_metric_view_sql(
    table_name: str, dims: List[str], measures: List[str]
) -> str:
    def q(n: str) -> str:
        return "`" + n.replace("`", "``") + "`"

    select_parts = [q(d) for d in dims] + [f"MEASURE({q(m)}) AS {q(m)}" for m in measures]
    if not select_parts:
        return f"SELECT * FROM {table_name}"
    sql = f"SELECT {', '.join(select_parts)} FROM {table_name}"
    if dims and measures:
        sql += " GROUP BY " + ", ".join(q(d) for d in dims)
    elif dims and not measures:
        sql = f"SELECT DISTINCT {', '.join(q(d) for d in dims)} FROM {table_name}"
    return sql

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    file_source = config.get("file_source")
    table_source = config.get("table_source")

    if file_source:
        path = file_source.get("path")
        if not path:
            raise ValueError("Source: 'path' is required for file source")
        options = []
        for key, value in file_source.items():
            if key == "path":
                continue
            if key == "headerRows" and isinstance(value, bool):
                options.append(f"{key}=>{1 if value else 0}")
            elif isinstance(value, bool):
                options.append(f'{key}=>{"true" if value else "false"}')
            elif isinstance(value, (int, float)):
                options.append(f"{key}=>{value}")
            else:
                clean = _strip_sql_quotes(str(value))
                if key == "dataAddress" and clean.startswith("!"):
                    clean = clean[1:]
                options.append(f'{key}=>"{clean}"')
        opts = ", ".join(options)
        sql = f'SELECT * FROM read_files("{path}", {opts})' if opts else f'SELECT * FROM read_files("{path}")'
        out = spark.sql(sql)
    elif table_source:
        table_name = table_source.get("tableName")
        if not table_name:
            raise ValueError("Source: 'tableName' is required for table source")

        mv_selection = table_source.get("metricView")
        if mv_selection is not None:
            dims = mv_selection.get("dimensions")
            measures = mv_selection.get("measures")
            if dims is None or measures is None:
                raise ValueError(
                    "Source: metricView selection is incomplete (both "
                    "'dimensions' and 'measures' must be explicit lists). "
                    "Re-open the source node and select the metric view "
                    "again to re-seed the picker."
                )
            sql = _build_metric_view_sql(table_name, list(dims), list(measures))
            out = spark.sql(sql)
        elif table_source.get("isExpression"):
            out = spark.sql(table_name)
        else:
            out = spark.table(table_name)
    else:
        raise ValueError("Source: either 'file_source' or 'table_source' must be configured")

    return {"data": out}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {
    "table_source": {
        "tableName": "lakeflow_designer.bronze_tables.order_items"
    }
}
inputs = {}
out = run(config, inputs, spark)
ctx["source_1.data"] = out["data"]

In [0]:
"""
id: source_14
template: source
templateVersion: 2.0.0
name: reviews
position:
  x: 1552.4015126818242
  y: 945.3917411054297
description:
  text: Load all data from reviews table.
  hash: 0a636f29
previewCodeHash: 6a03d69c2cafef3e
previewMode: "1000"
config:
  table_source:
    tableName: datamodeling.bronze.reviews
input: []
"""

# generated from the system
from typing import Dict, Any, List

def _strip_sql_quotes(s):
    if isinstance(s, str) and len(s) >= 2:
        if (s[0] == '"' and s[-1] == '"') or (s[0] == "'" and s[-1] == "'"):
            return s[1:-1]
    return s

def _build_metric_view_sql(
    table_name: str, dims: List[str], measures: List[str]
) -> str:
    def q(n: str) -> str:
        return "`" + n.replace("`", "``") + "`"

    select_parts = [q(d) for d in dims] + [f"MEASURE({q(m)}) AS {q(m)}" for m in measures]
    if not select_parts:
        return f"SELECT * FROM {table_name}"
    sql = f"SELECT {', '.join(select_parts)} FROM {table_name}"
    if dims and measures:
        sql += " GROUP BY " + ", ".join(q(d) for d in dims)
    elif dims and not measures:
        sql = f"SELECT DISTINCT {', '.join(q(d) for d in dims)} FROM {table_name}"
    return sql

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    file_source = config.get("file_source")
    table_source = config.get("table_source")

    if file_source:
        path = file_source.get("path")
        if not path:
            raise ValueError("Source: 'path' is required for file source")
        options = []
        for key, value in file_source.items():
            if key == "path":
                continue
            if key == "headerRows" and isinstance(value, bool):
                options.append(f"{key}=>{1 if value else 0}")
            elif isinstance(value, bool):
                options.append(f'{key}=>{"true" if value else "false"}')
            elif isinstance(value, (int, float)):
                options.append(f"{key}=>{value}")
            else:
                clean = _strip_sql_quotes(str(value))
                if key == "dataAddress" and clean.startswith("!"):
                    clean = clean[1:]
                options.append(f'{key}=>"{clean}"')
        opts = ", ".join(options)
        sql = f'SELECT * FROM read_files("{path}", {opts})' if opts else f'SELECT * FROM read_files("{path}")'
        out = spark.sql(sql)
    elif table_source:
        table_name = table_source.get("tableName")
        if not table_name:
            raise ValueError("Source: 'tableName' is required for table source")

        mv_selection = table_source.get("metricView")
        if mv_selection is not None:
            dims = mv_selection.get("dimensions")
            measures = mv_selection.get("measures")
            if dims is None or measures is None:
                raise ValueError(
                    "Source: metricView selection is incomplete (both "
                    "'dimensions' and 'measures' must be explicit lists). "
                    "Re-open the source node and select the metric view "
                    "again to re-seed the picker."
                )
            sql = _build_metric_view_sql(table_name, list(dims), list(measures))
            out = spark.sql(sql)
        elif table_source.get("isExpression"):
            out = spark.sql(table_name)
        else:
            out = spark.table(table_name)
    else:
        raise ValueError("Source: either 'file_source' or 'table_source' must be configured")

    return {"data": out}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {
    "table_source": {
        "tableName": "datamodeling.bronze.reviews"
    }
}
inputs = {}
out = run(config, inputs, spark)
ctx["source_14.data"] = out["data"]

In [0]:
"""
id: join_2
template: join
templateVersion: 1.0.0
name: ordersJoin_orderItems
position:
  x: 631
  y: 463.5
description:
  text: Left join on order_id and keep selected columns from both tables.
  hash: 9457e8db
previewCodeHash: 572497758033324f
previewMode: "1000"
config:
  join_type: left
  join_conditions: left.order_id = right.order_id
  expressions:
    - "`left`.order_id"
    - "`left`.customer_id"
    - "`left`.order_date"
    - "`left`.order_status"
    - "`right`.order_item_id"
    - "`right`.product_id"
    - "`right`.quantity"
    - "`right`.unit_price"
    - "`right`.discount_pct"
    - "`right`.line_total"
input:
  - node: source_0
    input_port: left
    output_port: data
  - node: source_1
    input_port: right
    output_port: data
"""

# generated from the system
from typing import Dict, Any, List
import pyspark.sql.functions as F

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    join_type = config.get("join_type", "inner").replace(" ", "_")
    join_condition = config.get("join_conditions", "")
    expressions: List[str] = config.get("expressions", [])

    df_left = inputs.get("left")
    df_right = inputs.get("right")
    if df_left is None or df_right is None:
        raise ValueError("Both left and right inputs must be connected")
    df_left = df_left.alias("left")
    df_right = df_right.alias("right")

    if not join_condition:
        result = df_left.join(df_right, how=join_type)
    else:
        result = df_left.join(df_right, F.expr(join_condition), how=join_type)

    if expressions:
        result = result.selectExpr(*expressions)

    return {"joined_data": result}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {
    "join_type": "left",
    "join_conditions": "left.order_id = right.order_id",
    "expressions": [
        "`left`.order_id",
        "`left`.customer_id",
        "`left`.order_date",
        "`left`.order_status",
        "`right`.order_item_id",
        "`right`.product_id",
        "`right`.quantity",
        "`right`.unit_price",
        "`right`.discount_pct",
        "`right`.line_total"
    ]
}
inputs = {
    "left": ctx["source_0.data"],
    "right": ctx["source_1.data"]
}
out = run(config, inputs, spark)
ctx["join_2.joined_data"] = out["joined_data"]

In [0]:
"""
id: ai_function_15
template: ai_function
templateVersion: 3.0.0
name: sentimental_analysis
position:
  x: 1873.2831755458892
  y: 942.8277778678118
description:
  text: Analyze sentiment of reviews and keep all columns.
  hash: 4044a671
previewCodeHash: 3c3bcf72ab5d174c
previewMode: "1000"
config:
  expressions:
    - ai_analyze_sentiment(review_body) `sentiment`
    - ai_translate('', '') `hindiTransaltion`
  keep_all_columns: true
input:
  - node: source_14
    input_port: data
    output_port: data
"""

# generated from the system
from typing import Dict, Any, List
import hashlib

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs["data"]

    # Table-valued AI functions (ai_forecast, ...) live in the FROM
    # clause and have no SELECT-list shape, so we run them via
    # spark.sql and bind the upstream DataFrame as a session-scoped
    # temp view substituted in for the
    # __lakebuilder_ai_function_input__ placeholder identifier
    # (which is a real SQL identifier so the persisted statement
    # round-trips through the SQL parser when the cell reloads).
    #
    # spark.sql returns a lazy DataFrame: the view name is resolved
    # by Spark at action time (preview limit/collect), not when
    # spark.sql is called. We therefore deliberately do NOT drop
    # the temp view here — dropping it would leave the lazy plan
    # pointing at a missing relation and trigger TABLE_OR_VIEW_NOT
    # _FOUND when the downstream action fires.
    #
    # The view name is derived deterministically from (tvf_sql,
    # id(df)) so reruns of the same cell reuse the same name and
    # createOrReplaceTempView keeps the session catalog bounded at
    # one entry per (cell × upstream) instead of growing one entry
    # per run. id(df) is included to keep two cells that happen to
    # have identical tvf_sql but distinct upstream DataFrames from
    # clobbering each other's bindings.
    tvf_sql: str = config.get("tvf_sql") or ""
    if tvf_sql:
        key = f"{tvf_sql}\x00{id(df)}".encode("utf-8")
        view_name = f"lakebuilder_ai_fn_{hashlib.sha256(key).hexdigest()[:12]}"
        df.createOrReplaceTempView(view_name)
        sql = tvf_sql.replace("__lakebuilder_ai_function_input__", view_name)
        return {"ai_data": spark.sql(sql)}

    expressions: List[str] = config.get("expressions", [])
    if not expressions:
        return {"ai_data": df}

    keep_all_columns: bool = config.get("keep_all_columns", True)
    if keep_all_columns:
        return {"ai_data": df.selectExpr(*expressions, "*")}
    return {"ai_data": df.selectExpr(*expressions)}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {
    "expressions": [
        "ai_analyze_sentiment(review_body) `sentiment`",
        "ai_translate('', '') `hindiTransaltion`"
    ],
    "keep_all_columns": True
}
inputs = {
    "data": ctx["source_14.data"]
}
out = run(config, inputs, spark)
ctx["ai_function_15.ai_data"] = out["ai_data"]

In [0]:
"""
id: join_5
template: join
templateVersion: 1.0.0
name: join_5
position:
  x: 905
  y: 520.5
description:
  text: Left join on customer_id to combine order data with customer details, keeping only relevant columns from both.
  hash: f89ef02b
previewCodeHash: 8be49202b4019cfa
previewMode: "1000"
config:
  join_type: left
  join_conditions: left.customer_id = right.customer_id
  expressions:
    - "`left`.order_id"
    - "`left`.customer_id `customer_id`"
    - "`left`.order_date"
    - "`left`.order_status"
    - "`left`.order_item_id"
    - "`left`.product_id"
    - "`left`.quantity"
    - "`left`.unit_price"
    - "`left`.discount_pct"
    - "`left`.line_total"
    - "`right`.customer_name"
    - "`right`.email"
    - "`right`.phone"
    - "`right`.city"
    - "`right`.state"
    - "`right`.country"
    - "`right`.created_date"
input:
  - node: join_2
    input_port: left
    output_port: joined_data
  - node: python_3
    input_port: right
    output_port: result
"""

# generated from the system
from typing import Dict, Any, List
import pyspark.sql.functions as F

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    join_type = config.get("join_type", "inner").replace(" ", "_")
    join_condition = config.get("join_conditions", "")
    expressions: List[str] = config.get("expressions", [])

    df_left = inputs.get("left")
    df_right = inputs.get("right")
    if df_left is None or df_right is None:
        raise ValueError("Both left and right inputs must be connected")
    df_left = df_left.alias("left")
    df_right = df_right.alias("right")

    if not join_condition:
        result = df_left.join(df_right, how=join_type)
    else:
        result = df_left.join(df_right, F.expr(join_condition), how=join_type)

    if expressions:
        result = result.selectExpr(*expressions)

    return {"joined_data": result}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {
    "join_type": "left",
    "join_conditions": "left.customer_id = right.customer_id",
    "expressions": [
        "`left`.order_id",
        "`left`.customer_id `customer_id`",
        "`left`.order_date",
        "`left`.order_status",
        "`left`.order_item_id",
        "`left`.product_id",
        "`left`.quantity",
        "`left`.unit_price",
        "`left`.discount_pct",
        "`left`.line_total",
        "`right`.customer_name",
        "`right`.email",
        "`right`.phone",
        "`right`.city",
        "`right`.state",
        "`right`.country",
        "`right`.created_date"
    ]
}
inputs = {
    "left": ctx["join_2.joined_data"],
    "right": ctx["python_3.result"]
}
out = run(config, inputs, spark)
ctx["join_5.joined_data"] = out["joined_data"]

In [0]:
"""
id: ai_function_17
template: ai_function
templateVersion: 3.0.0
name: translate
position:
  x: 2152.862081372492
  y: 1083.9119967024915
description:
  text: Return all columns without changes.
  hash: 2073bb2a
previewCodeHash: 49f3bed4a24a1c7a
previewMode: "1000"
config:
  expressions:
    - ai_translate(sentiment, 'Hindi') `hindiTranslation`
  keep_all_columns: true
input:
  - node: ai_function_15
    input_port: data
    output_port: ai_data
"""

# generated from the system
from typing import Dict, Any, List
import hashlib

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs["data"]

    # Table-valued AI functions (ai_forecast, ...) live in the FROM
    # clause and have no SELECT-list shape, so we run them via
    # spark.sql and bind the upstream DataFrame as a session-scoped
    # temp view substituted in for the
    # __lakebuilder_ai_function_input__ placeholder identifier
    # (which is a real SQL identifier so the persisted statement
    # round-trips through the SQL parser when the cell reloads).
    #
    # spark.sql returns a lazy DataFrame: the view name is resolved
    # by Spark at action time (preview limit/collect), not when
    # spark.sql is called. We therefore deliberately do NOT drop
    # the temp view here — dropping it would leave the lazy plan
    # pointing at a missing relation and trigger TABLE_OR_VIEW_NOT
    # _FOUND when the downstream action fires.
    #
    # The view name is derived deterministically from (tvf_sql,
    # id(df)) so reruns of the same cell reuse the same name and
    # createOrReplaceTempView keeps the session catalog bounded at
    # one entry per (cell × upstream) instead of growing one entry
    # per run. id(df) is included to keep two cells that happen to
    # have identical tvf_sql but distinct upstream DataFrames from
    # clobbering each other's bindings.
    tvf_sql: str = config.get("tvf_sql") or ""
    if tvf_sql:
        key = f"{tvf_sql}\x00{id(df)}".encode("utf-8")
        view_name = f"lakebuilder_ai_fn_{hashlib.sha256(key).hexdigest()[:12]}"
        df.createOrReplaceTempView(view_name)
        sql = tvf_sql.replace("__lakebuilder_ai_function_input__", view_name)
        return {"ai_data": spark.sql(sql)}

    expressions: List[str] = config.get("expressions", [])
    if not expressions:
        return {"ai_data": df}

    keep_all_columns: bool = config.get("keep_all_columns", True)
    if keep_all_columns:
        return {"ai_data": df.selectExpr(*expressions, "*")}
    return {"ai_data": df.selectExpr(*expressions)}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {
    "expressions": [
        "ai_translate(sentiment, 'Hindi') `hindiTranslation`"
    ],
    "keep_all_columns": True
}
inputs = {
    "data": ctx["ai_function_15.ai_data"]
}
out = run(config, inputs, spark)
ctx["ai_function_17.ai_data"] = out["ai_data"]

In [0]:
"""
id: workspace.default.output_16
template: output
templateVersion: 1.0.0
name: sentimentsResults
position:
  x: 2148.5738398930716
  y: 950.6127826591905
description:
  text: Save data to the specified table in overwrite mode.
  hash: ed1890d2
previewMode: "1000"
config:
  catalog: lakeflow_designer
  schema: silver_table
  table_name: sentiments
input:
  - node: ai_function_15
    input_port: data
    output_port: ai_data
"""

# generated from the system
from typing import Dict, Any

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs["data"]
    catalog = config.get("catalog", "")
    schema = config.get("schema", "")
    table_name = config.get("table_name", "")

    if not table_name:
        raise ValueError("Output: 'table_name' is required")

    parts = [p for p in [catalog, schema, table_name] if p]
    full_name = ".".join(parts)

    df.write.mode("overwrite").saveAsTable(full_name)

    return {}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {
    "catalog": "lakeflow_designer",
    "schema": "silver_table",
    "table_name": "sentiments"
}
inputs = {
    "data": ctx["ai_function_15.ai_data"]
}
out = run(config, inputs, spark)

In [0]:
"""
id: join_6
template: join
templateVersion: 1.0.0
name: oneBigTable
position:
  x: 1172
  y: 609.5
description:
  text: Left join on order_id with selected columns from both sides.
  hash: 6060c0af
previewCodeHash: 5c0724eaf6155c46
previewMode: "1000"
config:
  join_type: left
  join_conditions: left.order_id = right.order_id
  expressions:
    - "`left`.order_id `order_id`"
    - "`left`.customer_id"
    - "`left`.order_date"
    - "`left`.order_status"
    - "`left`.order_item_id"
    - "`left`.product_id"
    - "`left`.quantity"
    - "`left`.unit_price"
    - "`left`.discount_pct"
    - "`left`.line_total"
    - "`left`.customer_name"
    - "`left`.email"
    - "`left`.phone"
    - "`left`.city"
    - "`left`.state"
    - "`left`.country"
    - "`left`.created_date"
    - "`right`.shipment_id"
    - "`right`.ship_date"
    - "`right`.ship_mode"
    - "`right`.shipping_cost"
input:
  - node: join_5
    input_port: left
    output_port: joined_data
  - node: python_4
    input_port: right
    output_port: result
"""

# generated from the system
from typing import Dict, Any, List
import pyspark.sql.functions as F

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    join_type = config.get("join_type", "inner").replace(" ", "_")
    join_condition = config.get("join_conditions", "")
    expressions: List[str] = config.get("expressions", [])

    df_left = inputs.get("left")
    df_right = inputs.get("right")
    if df_left is None or df_right is None:
        raise ValueError("Both left and right inputs must be connected")
    df_left = df_left.alias("left")
    df_right = df_right.alias("right")

    if not join_condition:
        result = df_left.join(df_right, how=join_type)
    else:
        result = df_left.join(df_right, F.expr(join_condition), how=join_type)

    if expressions:
        result = result.selectExpr(*expressions)

    return {"joined_data": result}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {
    "join_type": "left",
    "join_conditions": "left.order_id = right.order_id",
    "expressions": [
        "`left`.order_id `order_id`",
        "`left`.customer_id",
        "`left`.order_date",
        "`left`.order_status",
        "`left`.order_item_id",
        "`left`.product_id",
        "`left`.quantity",
        "`left`.unit_price",
        "`left`.discount_pct",
        "`left`.line_total",
        "`left`.customer_name",
        "`left`.email",
        "`left`.phone",
        "`left`.city",
        "`left`.state",
        "`left`.country",
        "`left`.created_date",
        "`right`.shipment_id",
        "`right`.ship_date",
        "`right`.ship_mode",
        "`right`.shipping_cost"
    ]
}
inputs = {
    "left": ctx["join_5.joined_data"],
    "right": ctx["python_4.result"]
}
out = run(config, inputs, spark)
ctx["join_6.joined_data"] = out["joined_data"]

In [0]:
"""
id: aggregate_7
template: aggregate
templateVersion: 1.0.0
name: orderByCity
position:
  x: 1470.1171720925854
  y: 624.2058070588598
description:
  text: Group by city and calculate total orders and rounded total amount.
  hash: cce8cce4
previewCodeHash: 36590dcaeeb5d1e9
previewMode: "1000"
config:
  group_bys:
    - expr: city
      type: column
  aggregations:
    - columnExpr:
        expr: order_id
        type: column
      fn: COUNT
      alias: TotalOrders
    - columnExpr:
        expr: ROUND(SUM(unit_price),2)
        type: expr
      fn: "-"
      alias: TotalAmount
input:
  - node: join_6
    input_port: data
    output_port: joined_data
"""

# generated from the system
import math
from typing import Dict, Any
import pyspark.sql.functions as F

DEFAULT_PERCENTILE = 0.5

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs.get("data")
    group_bys = config.get("group_bys", [])
    aggregations = config.get("aggregations", [])

    group_by_set = set(e for gb in group_bys if (e := gb.get("expr", "")))

    agg_exprs = []
    for agg_def in aggregations:
        col_expr = agg_def.get("columnExpr", {})
        raw_expr = col_expr.get("expr", "")
        fn = agg_def.get("fn", "-")
        alias = agg_def.get("alias")

        if (fn == "-" or fn == "_") and not alias and raw_expr in group_by_set:
            continue

        fn_map = {
            "SUM": F.sum,
            "AVG": F.avg,
            "COUNT": F.count,
            "MIN": F.min,
            "MAX": F.max,
            "MEAN": F.mean,
            "MEDIAN": F.median,
            "STDDEV": F.stddev,
            "VARIANCE": F.variance,
        }

        agg_fn = fn_map.get(fn)
        if agg_fn:
            col = agg_fn(raw_expr)
        elif fn == "-" or fn == "_":
            col = F.expr(raw_expr) if col_expr.get("type") == "expr" else F.col(raw_expr)
        elif fn == "PERCENTILE":
            raw_pct = agg_def.get("percentage")
            if (
                isinstance(raw_pct, (int, float))
                and not isinstance(raw_pct, bool)
                and math.isfinite(raw_pct)
            ):
                pct = max(0.0, min(1.0, float(raw_pct)))
            else:
                pct = DEFAULT_PERCENTILE
            col = F.expr(f"PERCENTILE({raw_expr}, {pct})")
        else:
            col = F.expr(f"{fn}({raw_expr})")

        if alias:
            col = col.alias(alias)

        agg_exprs.append(col)

    group_cols = [
        gb.get("expr", "") for gb in group_bys if gb.get("expr", "")
    ]

    if not agg_exprs:
        if group_cols:
            result = df.select(*group_cols).distinct()
            return {"aggregated_data": result}
        return {"aggregated_data": df}

    if group_cols:
        result = df.groupBy(*group_cols).agg(*agg_exprs)
    else:
        result = df.agg(*agg_exprs)

    return {"aggregated_data": result}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {
    "group_bys": [
        {
            "expr": "city",
            "type": "column"
        }
    ],
    "aggregations": [
        {
            "columnExpr": {
                "expr": "order_id",
                "type": "column"
            },
            "fn": "COUNT",
            "alias": "TotalOrders",
            "withAsKeyword": None
        },
        {
            "columnExpr": {
                "expr": "ROUND(SUM(unit_price),2)",
                "type": "expr"
            },
            "fn": "-",
            "alias": "TotalAmount",
            "withAsKeyword": None
        }
    ]
}
inputs = {
    "data": ctx["join_6.joined_data"]
}
out = run(config, inputs, spark)
ctx["aggregate_7.aggregated_data"] = out["aggregated_data"]

In [0]:
"""
id: transform_order_status_month
template: transform
templateVersion: 1.0.0
name: transform_order_status_month
position:
  x: 1372
  y: 776
description:
  text: Keep order status, id, date, and add order month.
  hash: 6181359c
previewCodeHash: b5d8d6a99dfbbd47
previewMode: "1000"
config:
  expressions:
    - order_status
    - order_id
    - order_date
    - MONTH(order_date) AS `order_month`
input:
  - node: join_6
    input_port: data
    output_port: joined_data
"""

# generated from the system
from typing import Dict, Any, List

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs["data"]
    expressions: List[str] = config.get("expressions", [])

    if not expressions:
        return {"transformed_data": df}

    return {"transformed_data": df.selectExpr(*expressions)}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {
    "expressions": [
        "order_status",
        "order_id",
        "order_date",
        "MONTH(order_date) AS `order_month`"
    ]
}
inputs = {
    "data": ctx["join_6.joined_data"]
}
out = run(config, inputs, spark)
ctx["transform_order_status_month.transformed_data"] = out["transformed_data"]

In [0]:
"""
id: sort_8
template: sort
templateVersion: 1.0.0
name: sorted
position:
  x: 1766.574125963972
  y: 627.0101881258896
description:
  text: Sort data by TotalAmount in descending order.
  hash: 8120b041
previewCodeHash: 22363c9345488f80
previewMode: "1000"
config:
  sort_expressions:
    - columnExpr:
        expr: TotalAmount
        type: column
      sortBy: DESC
input:
  - node: aggregate_7
    input_port: data
    output_port: aggregated_data
"""

# generated from the system
from typing import Dict, Any
import pyspark.sql.functions as F

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs.get("data")
    sort_expressions = config.get("sort_expressions", [])

    if not sort_expressions:
        return {"sorted_data": df}

    order_cols = []
    for sort_def in sort_expressions:
        col_expr = sort_def.get("columnExpr", {})
        raw_expr = col_expr.get("expr", "")
        direction = sort_def.get("sortBy", "UNSET")

        col = F.col(raw_expr)
        if direction == "DESC":
            col = col.desc()
        elif direction == "ASC":
            col = col.asc()

        order_cols.append(col)

    return {"sorted_data": df.orderBy(*order_cols)}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {
    "sort_expressions": [
        {
            "columnExpr": {
                "expr": "TotalAmount",
                "type": "column"
            },
            "sortBy": "DESC"
        }
    ]
}
inputs = {
    "data": ctx["aggregate_7.aggregated_data"]
}
out = run(config, inputs, spark)
ctx["sort_8.sorted_data"] = out["sorted_data"]

In [0]:
"""
id: aggregate_order_status_month
template: aggregate
templateVersion: 1.0.0
name: aggregate_order_status_month
position:
  x: 1592
  y: 776
description:
  text: Group data by order status and month, then count total orders for each group.
  hash: ba2e98ef
previewCodeHash: 700eb6998097a805
previewMode: "1000"
config:
  group_bys:
    - expr: order_status
      type: expr
    - expr: order_month
      type: expr
  aggregations:
    - columnExpr:
        expr: order_id
        type: expr
      fn: COUNT
      alias: TotalOrders
input:
  - node: transform_order_status_month
    input_port: data
    output_port: transformed_data
"""

# generated from the system
import math
from typing import Dict, Any
import pyspark.sql.functions as F

DEFAULT_PERCENTILE = 0.5

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs.get("data")
    group_bys = config.get("group_bys", [])
    aggregations = config.get("aggregations", [])

    group_by_set = set(e for gb in group_bys if (e := gb.get("expr", "")))

    agg_exprs = []
    for agg_def in aggregations:
        col_expr = agg_def.get("columnExpr", {})
        raw_expr = col_expr.get("expr", "")
        fn = agg_def.get("fn", "-")
        alias = agg_def.get("alias")

        if (fn == "-" or fn == "_") and not alias and raw_expr in group_by_set:
            continue

        fn_map = {
            "SUM": F.sum,
            "AVG": F.avg,
            "COUNT": F.count,
            "MIN": F.min,
            "MAX": F.max,
            "MEAN": F.mean,
            "MEDIAN": F.median,
            "STDDEV": F.stddev,
            "VARIANCE": F.variance,
        }

        agg_fn = fn_map.get(fn)
        if agg_fn:
            col = agg_fn(raw_expr)
        elif fn == "-" or fn == "_":
            col = F.expr(raw_expr) if col_expr.get("type") == "expr" else F.col(raw_expr)
        elif fn == "PERCENTILE":
            raw_pct = agg_def.get("percentage")
            if (
                isinstance(raw_pct, (int, float))
                and not isinstance(raw_pct, bool)
                and math.isfinite(raw_pct)
            ):
                pct = max(0.0, min(1.0, float(raw_pct)))
            else:
                pct = DEFAULT_PERCENTILE
            col = F.expr(f"PERCENTILE({raw_expr}, {pct})")
        else:
            col = F.expr(f"{fn}({raw_expr})")

        if alias:
            col = col.alias(alias)

        agg_exprs.append(col)

    group_cols = [
        gb.get("expr", "") for gb in group_bys if gb.get("expr", "")
    ]

    if not agg_exprs:
        if group_cols:
            result = df.select(*group_cols).distinct()
            return {"aggregated_data": result}
        return {"aggregated_data": df}

    if group_cols:
        result = df.groupBy(*group_cols).agg(*agg_exprs)
    else:
        result = df.agg(*agg_exprs)

    return {"aggregated_data": result}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {
    "group_bys": [
        {
            "expr": "order_status",
            "type": "expr"
        },
        {
            "expr": "order_month",
            "type": "expr"
        }
    ],
    "aggregations": [
        {
            "columnExpr": {
                "expr": "order_id",
                "type": "expr"
            },
            "fn": "COUNT",
            "alias": "TotalOrders",
            "withAsKeyword": None
        }
    ]
}
inputs = {
    "data": ctx["transform_order_status_month.transformed_data"]
}
out = run(config, inputs, spark)
ctx["aggregate_order_status_month.aggregated_data"] = out["aggregated_data"]

In [0]:
"""
id: workspace.default.output_9
template: output
templateVersion: 1.0.0
name: AggOrders
position:
  x: 2041.9982218326354
  y: 627.0101881258896
description:
  text: Save data as a table, replacing existing data if needed.
  hash: 15ab86ba
previewMode: "1000"
config:
  catalog: lakeflow_designer
  schema: silver_table
  table_name: AggOrders
input:
  - node: sort_8
    input_port: data
    output_port: sorted_data
"""

# generated from the system
from typing import Dict, Any

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs["data"]
    catalog = config.get("catalog", "")
    schema = config.get("schema", "")
    table_name = config.get("table_name", "")

    if not table_name:
        raise ValueError("Output: 'table_name' is required")

    parts = [p for p in [catalog, schema, table_name] if p]
    full_name = ".".join(parts)

    df.write.mode("overwrite").saveAsTable(full_name)

    return {}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {
    "catalog": "lakeflow_designer",
    "schema": "silver_table",
    "table_name": "AggOrders"
}
inputs = {
    "data": ctx["sort_8.sorted_data"]
}
out = run(config, inputs, spark)

In [0]:
"""
id: sort_order_status_month
template: sort
templateVersion: 1.0.0
name: sort_order_status_month
position:
  x: 1810
  y: 776
description:
  text: Sort data by order_month in descending order.
  hash: fc3a5dae
previewCodeHash: 89d4261b9cbbfa4b
previewMode: "1000"
config:
  sort_expressions:
    - columnExpr:
        expr: order_month
        type: expr
      sortBy: DESC
input:
  - node: aggregate_order_status_month
    input_port: data
    output_port: aggregated_data
"""

# generated from the system
from typing import Dict, Any
import pyspark.sql.functions as F

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs.get("data")
    sort_expressions = config.get("sort_expressions", [])

    if not sort_expressions:
        return {"sorted_data": df}

    order_cols = []
    for sort_def in sort_expressions:
        col_expr = sort_def.get("columnExpr", {})
        raw_expr = col_expr.get("expr", "")
        direction = sort_def.get("sortBy", "UNSET")

        col = F.col(raw_expr)
        if direction == "DESC":
            col = col.desc()
        elif direction == "ASC":
            col = col.asc()

        order_cols.append(col)

    return {"sorted_data": df.orderBy(*order_cols)}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {
    "sort_expressions": [
        {
            "columnExpr": {
                "expr": "order_month",
                "type": "expr"
            },
            "sortBy": "DESC"
        }
    ]
}
inputs = {
    "data": ctx["aggregate_order_status_month.aggregated_data"]
}
out = run(config, inputs, spark)
ctx["sort_order_status_month.sorted_data"] = out["sorted_data"]

In [0]:
"""
id: workspace.default.output_13
template: output
templateVersion: 1.0.0
name: lakeflow_designer.silver_table.orderStatus
position:
  x: 2065.5265397713806
  y: 789.889140758933
description:
  text: Save data as a new table, overwriting existing one if any.
  hash: 50817b95
previewMode: "1000"
config:
  catalog: lakeflow_designer
  schema: silver_table
  table_name: orderStatus
input:
  - node: sort_order_status_month
    input_port: data
    output_port: sorted_data
"""

# generated from the system
from typing import Dict, Any

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs["data"]
    catalog = config.get("catalog", "")
    schema = config.get("schema", "")
    table_name = config.get("table_name", "")

    if not table_name:
        raise ValueError("Output: 'table_name' is required")

    parts = [p for p in [catalog, schema, table_name] if p]
    full_name = ".".join(parts)

    df.write.mode("overwrite").saveAsTable(full_name)

    return {}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {
    "catalog": "lakeflow_designer",
    "schema": "silver_table",
    "table_name": "orderStatus"
}
inputs = {
    "data": ctx["sort_order_status_month.sorted_data"]
}
out = run(config, inputs, spark)